# Stage 3C: Mechanistic Intervention (Frequency=1)

This notebook tests the hypothesis that cached inverse-root staleness is the dominant source of instability. We set `precondition_frequency=1` to completely remove staleness while holding all other factors constant.

**Renaming Note:**
The optimizer is now named `PaperStyleSOAP` to scientifically acknowledge that this is a reconstruction of the paper's style, given that some numerical implementation details (like exact EMA initialization and damping scale) are not uniquely specified in the source text.

In [ ]:
import json
import math
import time
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
class PaperStyleSOAP(optim.Optimizer):
    """Paper-style reconstruction of the SOAP preconditioner with internal MoQ computation and diagnostics."""
    def __init__(self, params, lr=1e-3, beta2=0.95, damping=1e-3, precondition_frequency=1, max_precond_dim=10000, mu=0.0):
        if lr <= 0: raise ValueError('lr must be positive')
        if not 0.0 <= mu < 1.0: raise ValueError('mu must be in [0, 1)')
        defaults = dict(lr=lr, beta2=beta2, damping=damping, precondition_frequency=precondition_frequency, max_precond_dim=max_precond_dim, mu=mu)
        super().__init__(params, defaults)
        self.layer_diagnostics = []

    @staticmethod
    def _matrix_view(gradient):
        if gradient.ndim < 2: return None
        rows = gradient.shape[0]
        columns = gradient.numel() // rows
        return gradient.reshape(rows, columns)

    @staticmethod
    def _inverse_sqrt(matrix, damping):
        original_dtype = matrix.dtype
        matrix64 = matrix.to(torch.float64)
        matrix64 = 0.5 * (matrix64 + matrix64.T) # Symmetrize to prevent numerical artifacts
        identity = torch.eye(matrix64.shape[0], device=matrix64.device, dtype=torch.float64)
        try:
            eigenvalues, eigenvectors = torch.linalg.eigh(matrix64 + damping * identity)
            eigenvalues = torch.clamp(eigenvalues, min=damping)
            inverse_sqrt = torch.rsqrt(eigenvalues)
            result = eigenvectors @ torch.diag(inverse_sqrt) @ eigenvectors.T
            return result.to(original_dtype)
        except Exception:
            return torch.eye(matrix64.shape[0], device=matrix.device, dtype=original_dtype)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
                
        self.layer_diagnostics = []

        for group in self.param_groups:
            lr = group['lr']
            beta2 = group['beta2']
            damping = group['damping']
            frequency = group['precondition_frequency']
            mu = group['mu']

            for layer_idx, parameter in enumerate(group['params']):
                if parameter.grad is None:
                    continue

                raw_gradient = parameter.grad
                state = self.state[parameter]

                if len(state) == 0:
                    state['step'] = 0
                    state['momentum_buffer'] = torch.zeros_like(parameter)
                    state['prev_grad'] = torch.zeros_like(raw_gradient)
                    if raw_gradient.ndim < 2:
                        state['mode'] = 'diagonal'
                        state['v'] = torch.zeros_like(raw_gradient)
                    else:
                        matrix = self._matrix_view(raw_gradient)
                        rows, columns = matrix.shape
                        if max(rows, columns) > group['max_precond_dim']:
                            state['mode'] = 'diagonal'
                            state['v'] = torch.zeros_like(raw_gradient)
                        else:
                            state['mode'] = 'structured'
                            state['A'] = torch.zeros((rows, rows), device=raw_gradient.device, dtype=raw_gradient.dtype)
                            state['B'] = torch.zeros((columns, columns), device=raw_gradient.device, dtype=raw_gradient.dtype)
                            state['inv_sqrt_A'] = None
                            state['inv_sqrt_B'] = None
                            
                state['step'] += 1
                step = state['step']

                if step == 1 or mu == 0.0:
                    g_moq = raw_gradient.clone()
                else:
                    g_moq = (1 + mu) * raw_gradient - mu * state['prev_grad']
                
                state['prev_grad'].copy_(raw_gradient)

                if state['mode'] == 'diagonal':
                    state['v'].mul_(beta2).addcmul_(g_moq, g_moq, value=1.0 - beta2)
                    preconditioned = g_moq / torch.sqrt(state['v'] + damping)
                else:
                    matrix_g_moq = self._matrix_view(g_moq)
                    matrix_raw = self._matrix_view(raw_gradient)
                    
                    # 1. Matched preconditioner computation
                    matched_A = matrix_g_moq @ matrix_g_moq.T
                    matched_B = matrix_g_moq.T @ matrix_g_moq
                    matched_inv_A = self._inverse_sqrt(matched_A, damping)
                    matched_inv_B = self._inverse_sqrt(matched_B, damping)
                    matched_precond = matched_inv_A @ matrix_g_moq @ matched_inv_B
                    
                    # Update EMA factors
                    factor_outer_left = matched_A
                    factor_outer_right = matched_B
                    state['A'].mul_(beta2).add_(factor_outer_left, alpha=1.0 - beta2)
                    state['B'].mul_(beta2).add_(factor_outer_right, alpha=1.0 - beta2)
                    
                    # 2. EMA-fresh preconditioner computation
                    ema_fresh_inv_A = self._inverse_sqrt(state['A'], damping)
                    ema_fresh_inv_B = self._inverse_sqrt(state['B'], damping)
                    ema_fresh_precond = ema_fresh_inv_A @ matrix_g_moq @ ema_fresh_inv_B
                    
                    # 3. EMA-stale preconditioner computation
                    if state['inv_sqrt_A'] is None or step % frequency == 0:
                        state['inv_sqrt_A'] = ema_fresh_inv_A
                        state['inv_sqrt_B'] = ema_fresh_inv_B

                    ema_stale_precond = (
                        state['inv_sqrt_A']
                        @ matrix_g_moq
                        @ state['inv_sqrt_B']
                    )
                    preconditioned = ema_stale_precond.reshape_as(raw_gradient)

                    # DIAGNOSTICS FOR MECHANISTIC AUDIT
                    g_moq_norm = float(torch.linalg.vector_norm(matrix_g_moq))
                    raw_norm = float(torch.linalg.vector_norm(matrix_raw))
                    
                    matched_norm = float(torch.linalg.vector_norm(matched_precond))
                    ema_fresh_norm = float(torch.linalg.vector_norm(ema_fresh_precond))
                    ema_stale_norm = float(torch.linalg.vector_norm(ema_stale_precond))
                    
                    diag = {
                        "layer": layer_idx,
                        "raw_grad_norm": raw_norm,
                        "MoQ_raw_ratio": (g_moq_norm / (raw_norm + 1e-12)) if mu > 0.0 else 1.0,
                        "precond_norm": ema_stale_norm,
                        "precond_gain": ema_stale_norm / (g_moq_norm + 1e-12),
                        "matched_gain": matched_norm / (g_moq_norm + 1e-12),
                        "ema_fresh_gain": ema_fresh_norm / (g_moq_norm + 1e-12),
                        "ema_stale_gain": ema_stale_norm / (g_moq_norm + 1e-12),
                    }
                    self.layer_diagnostics.append(diag)

                buf = state['momentum_buffer']
                buf.mul_(mu).add_(preconditioned, alpha=-lr)
                parameter.add_(buf)

        return loss


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None

    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule


In [ ]:
def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def run_intervention(K: int, steps: int, use_moq: bool, mu: float = 0.85, seed: int = 0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule.pt")
    if schedule_path.exists():
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
    else:
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    lr = 1e-3
    opt_mu = mu if use_moq else 0.0
    # INTERVENTION: precondition_frequency = 1
    opt = PaperStyleSOAP(model.parameters(), lr=lr, beta2=0.95, damping=1e-3, precondition_frequency=1, mu=opt_mu)

    history = []
    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        opt.step()
        
        err = relative_l2(model, device)
        
        step_data = {
            "step": step,
            "loss": float(loss.detach()),
            "rel_l2": float(err),
            "layers": opt.layer_diagnostics
        }
        history.append(step_data)
        
        print(f"Step {step} | Loss {loss.item():.4e} | RelL2 {err:.4e}")

    return history


### Run Stage 3C Intervention (20 steps each, frequency=1)

In [ ]:
print("Running K=1 SOAP (frequency=1)...")
audit_K1_soap = run_intervention(K=1, steps=20, use_moq=False)
with open("stage3c_audit_K1_soap.json", "w") as f:
    json.dump(audit_K1_soap, f, indent=2)


In [ ]:
print("\nRunning K=1 SOAP+MoQ (frequency=1)...")
audit_K1_moq = run_intervention(K=1, steps=20, use_moq=True)
with open("stage3c_audit_K1_moq.json", "w") as f:
    json.dump(audit_K1_moq, f, indent=2)


In [ ]:
print("\nRunning K=20 SOAP (frequency=1)...")
audit_K20_soap = run_intervention(K=20, steps=20, use_moq=False)
with open("stage3c_audit_K20_soap.json", "w") as f:
    json.dump(audit_K20_soap, f, indent=2)


In [ ]:
print("\nRunning K=20 SOAP+MoQ (frequency=1)...")
audit_K20_moq = run_intervention(K=20, steps=20, use_moq=True)
with open("stage3c_audit_K20_moq.json", "w") as f:
    json.dump(audit_K20_moq, f, indent=2)
